# Classification Modeling

In [158]:
import pandas as pd
from trial import get_columns_by_type
from preprocessing import percentage_to_int, join_and_sort
from pathlib import Path
import shutil
from category_encoders import OneHotEncoder, TargetEncoder
from joblib import load, dump
import pandas as pd
import numpy as np

from sklearn import set_config
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import make_scorer, mean_absolute_error, confusion_matrix, classification_report, precision_score, recall_score, f1_score, accuracy_score

from transformers import DateTransformer
from scipy.stats import f
set_config(transform_output="pandas")

## Load Data and Train Test Split

In [110]:
# load data
df = pd.read_csv('../data/preprocessed.csv', index_col="fixture_id")
df.drop("Unnamed: 0", axis=1, inplace=True)
df['start_time'] = pd.to_datetime(df['start_time'])
df.drop("passing_accuracy", axis=1, inplace=True)
col_types = get_columns_by_type(df)
df['start_time'].max()

Timestamp('2023-05-28 11:30:00')

## make target variables categorical

In [111]:
def categorize(num_goals):
    if num_goals < 0.5:
        return 0
    elif num_goals < 1.5:
        return 1
    elif num_goals < 2.5:
        return 2
    return 3

In [112]:
df['home_goals'] = df['home_goals'].apply(categorize)
df['away_goals'] = df['away_goals'].apply(categorize)

In [113]:
# Create training and testing sets
X = df.drop(['home_goals', 'away_goals'], axis=1)
home_goals = df['home_goals'].to_numpy()
away_goals = df['away_goals'].to_numpy()
X_train, X_test, y_home_train, y_home_test, y_away_train, y_away_test = train_test_split(X, home_goals, away_goals, test_size=0.25, shuffle=False)

# Fitting A Model

In [141]:
# Defining our Pipiline
mae_scorer = make_scorer(mean_absolute_error)

date_transformer = Pipeline(steps=[
        ('transformer', DateTransformer()),
        ('encoder', TargetEncoder())])

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

feature_preprocessor = ColumnTransformer(transformers=[
    ('numerical', numeric_transformer, col_types["numeric"]),
    ('datetime', date_transformer, col_types["timestamp"])
])

params = {
    'estimator__C': [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
}

home_pipeline = Pipeline(steps=[
    ('preprocessor', feature_preprocessor),
    ('estimator', LogisticRegression())
])

svc_pipeline = Pipeline(steps=[
    ('preprocessor', feature_preprocessor),
    ('estimator', SVC(probability=True))
])

home_model = GridSearchCV(home_pipeline, param_grid=params)
svc_model = GridSearchCV(svc_pipeline, param_grid=params)

In [142]:
home_model.fit(X_train, y_home_train)
svc_model.fit(X_train, y_home_train)

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning: Skipping features without any observed values: ['home_std_rating' 'home_max_rating' 'away_max_rating' 'away_std_rating'
 'away_mean_rating' 'away_min_rating' 'home_min_rating' 'rating'
 'home_mean_rating']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/robertcampbell/sqlalchem

GridSearchCV(estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('numerical',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['away_max_fouls_drawn',
                                                                          'home_std_fouls_drawn',
                                                                          'home_top_scorer_5',
                                                                          'away_top_assistor_2',
                                                                          'home_top_scorer_11',
                                                                          'home_min_passing_accuracy',
                                                                          'away_max_interceptio...
                                                                          'home_std_dribble_success_percentage',
                                                                          'home_std_duels_won_percentage',
                                                                          'away_top_assistor_1',
                                                                          'away_top_assistor_7',
                                                                          'home_min_fouls_drawn',
                                                                          'away_top_assistor_4',
                                                                          'CUM_HT_HL', ...]),
                                                                        ('datetime',
                                                                         Pipeline(steps=[('transformer',
                                                                                          DateTransformer()),
                                                                                         ('encoder',
                                                                                          TargetEncoder())]),
                                                                         ['start_time'])])),
                                       ('estimator', SVC(probability=True))]),
             param_grid={'estimator__C': [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]})

In [143]:
home_pred = svc_model.predict(X_test)

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning: Skipping features without any observed values: ['home_std_rating' 'home_max_rating' 'away_max_rating' 'away_std_rating'
 'away_mean_rating' 'away_min_rating' 'home_min_rating' 'rating'
 'home_mean_rating']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [174]:
home_proba = home_model.predict_proba(X_test)
svc_proba = svc_model.predict_proba(X_test)

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning: Skipping features without any observed values: ['home_std_rating' 'home_max_rating' 'away_max_rating' 'away_std_rating'
 'away_mean_rating' 'away_min_rating' 'home_min_rating' 'rating'
 'home_mean_rating']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning: Skipping features without any observed values: ['home_std_rating' 'home_max_rating' 'away_max_rating' 'away_std_rating'
 'away_mean_rating' 'away_min_rating' 'home_min_rating' 'rating'
 'home_mean_rating']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [148]:
# make binary classifiers
def make_binary_classifiers(y_true: pd.Series, y_pred: pd.Series, threshold: int = 0, over_under: str = 'over'):
    def classify(data):
        if over_under == 'over':
            if data > threshold: 
                return 1
            return 0
        
        if data <= threshold:
            return 1
        else:
            return 0

    return y_true.apply(classify), y_pred.apply(classify)

In [149]:
home_pred

array([3, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 3, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1,
       1, 1, 1, 1, 1, 3, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
       0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 3, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1,
       1, 0, 1, 1, 2, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 3, 1, 1, 0, 2, 1, 1, 2, 1, 1, 1, 1, 0,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 3, 1, 1,
       1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 2, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 3, 1, 1, 1,
       0, 1, 1, 1, 1, 2, 1, 1, 0, 0, 1, 0, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 2, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       1, 1, 1, 3, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1,

In [150]:
THRESHOLD = 0
OVER = 'under'

In [151]:
# make binary classifiers for all bets
actual_home_is_under_0_5, predicted_home_is_under_0_5 = make_binary_classifiers(pd.Series(y_home_test), pd.Series(home_pred), threshold=0, over_under='under')
actual_home_is_over_0_5, predicted_home_is_over_0_5 = make_binary_classifiers(pd.Series(y_home_test), pd.Series(home_pred), threshold=0, over_under='over')
actual_home_is_under_1_5, predicted_home_is_under_1_5 = make_binary_classifiers(pd.Series(y_home_test), pd.Series(home_pred), threshold=1, over_under='under')
actual_home_is_over_1_5, predicted_home_is_over_1_5 = make_binary_classifiers(pd.Series(y_home_test), pd.Series(home_pred), threshold=1, over_under='over')
actual_home_is_under_2_5, predicted_home_is_under_2_5 = make_binary_classifiers(pd.Series(y_home_test), pd.Series(home_pred), threshold=2, over_under='under')
actual_home_is_over_2_5, predicted_home_is_over_2_5 = make_binary_classifiers(pd.Series(y_home_test), pd.Series(home_pred), threshold=2, over_under='over')

In [152]:
actual_home_is_under_1_5.count() - actual_home_is_under_1_5.sum()

341

## Get Metrics

In [153]:
# get metrics for multiclass classification --> this will determine how good our models are performing

In [154]:
def get_confusion_df(y_true, y_pred):
    """ creates a confusion matrix dataframe with appropriate axis labels """
    matrix_data = confusion_matrix(y_true, y_pred)
    index_tuples = [('actual', 0), ('actual', 1)]
    col_tuples = [('predicted', 0), ('predicted', 1)]
    idx = pd.MultiIndex.from_tuples(index_tuples)
    cols = pd.MultiIndex.from_tuples(col_tuples)
    confusion_df = pd.DataFrame(matrix_data, index=idx, columns=cols)
    return confusion_df

In [168]:
confusion_df = get_confusion_df(actual_home_is_over_2_5, predicted_home_is_over_2_5)
confusion_df

predicted    
                 0   1
actual 0       583  17
       1       150  11

# precision = True Positives / True Positives + False Positives

In [170]:
precision = precision_score(actual_home_is_over_2_5, predicted_home_is_over_2_5, average='binary')
print(precision)
recall = recall_score(actual_home_is_over_2_5, predicted_home_is_over_2_5, average='binary')
print(recall)
f1 = f1_score(actual_home_is_over_2_5, predicted_home_is_over_2_5, average='binary')
print(f1)


0.39285714285714285
0.06832298136645963
0.1164021164021164


In [171]:
accuracy_score(actual_home_is_under_2_5, predicted_home_is_under_2_5)

0.7805519053876478

In [183]:
X_test.iloc[0]

index                                     2281
rating                                     NaN
accurate_passes                            NaN
start_time                 2021-05-23 11:00:00
home_name                            Liverpool
                                  ...         
away_assists_0.5_quant                     1.0
away_assists_0.75_quant                    2.0
club_history_home_wins                       9
club_history_away_wins                       2
club_history_draws                           0
Name: 592871, Length: 177, dtype: object

In [184]:
y_home_test[0]

2

In [209]:
SAMPLE_IDX = 3

In [243]:
sample_home_name, sample_away_name = X_test["home_name"].iloc[SAMPLE_IDX], X_test["away_name"].iloc[SAMPLE_IDX]
sample = svc_proba[SAMPLE_IDX]
class_labels = ['hg < 0.5', '0.5 < hg < 1.5', '1.5 < hg < 2.5', 'hg > 2.5']
decision_labels = ['0.5', '1.5', '2.5']

In [244]:
import plotly.express as px
import plotly.graph_objects  as go

In [245]:
fig = px.bar(x=class_labels, y=sample)
fig.update_layout(
    title=f"{sample_home_name} vs {sample_away_name} Goal Probability Distribution for Home Team",
    xaxis=dict(title=f"Number of goals for {sample_home_name}"),
    yaxis=dict(title=f"Probability", tickformat=",.0%"),
)

In [247]:
under_probs = np.cumsum(sample[:3])
over_probs = np.array([1, 1, 1]) - under_probs

In [261]:
fig = go.Figure()

fig.add_trace(go.Bar(x=decision_labels[:3], y=under_probs, name="under", text=[str(round(prob * 100)) + "%" for prob in under_probs]))
fig.add_trace(go.Bar(x=decision_labels[:3], y=over_probs, name="over", text=[str(round(prob * 100)) + "%" for prob in over_probs]))
fig.update_layout(
    title=f"{sample_home_name} vs {sample_away_name} CDF for Home Team",
    xaxis=dict(title=f"Number of goals for {sample_home_name}"),
    yaxis=dict(title=f"Cummulative Probability", tickformat=",.0%"),
    barmode="stack",
    width=750
)

## Expected Value Calculation

Lets say johnny wants to know the expected value for betting over 1.5 for this game
E(v) = Probability of success * Return - Probability of Loss * loss

In [ ]:
expected_value = 

In [264]:
lg: LogisticRegression = home_model.best_estimator_["estimator"]

In [266]:
lg.coef_.shape

(4, 164)

In [267]:
from sklearn.inspection import permutation_importance

result = permutation_importance(home_model, X_test, y_home_test, n_repeats=10, random_state=42)


feature_importance = pd.DataFrame({'Feature': X.columns,
                                   'Importance': result.importances_mean,
                                   'Standard Deviation': result.importances_std})
feature_importance = feature_importance.sort_values('Importance', ascending=True)

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning:

Skipping features without any observed values: ['home_std_rating' 'home_max_rating' 'away_max_rating' 'away_std_rating'
 'away_mean_rating' 'away_min_rating' 'home_min_rating' 'rating'
 'home_mean_rating']. At least one non-missing value is needed for imputation with strategy='median'.

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning:

Skipping features without any observed values: ['home_std_rating' 'home_max_rating' 'away_max_rating' 'away_std_rating'
 'away_mean_rating' 'away_min_rating' 'home_min_rating' 'rating'
 'home_mean_rating']. At least one non-missing value is needed for imputation with strategy='median'.

/Users/robertcampbell/sqlalchemy-tutorial/venv/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning:

Skipping features without any observed values: ['home_std_rating' '

In [269]:
feature_importance.head(10)

,Feature,Importance,Standard Deviation
17,CUM_HT_AW,-0.014717,0.005315
42,home_mean_blocks,-0.014192,0.003889
50,home_mean_dribble_success_percentage,-0.012615,0.009819
117,home_top_scorer_4,-0.010907,0.004629
132,home_top_assistor_4,-0.010775,0.004350
38,home_mean_tackles,-0.010644,0.003022
82,away_mean_blocks,-0.010118,0.008436
172,away_assists_0.5_quant,-0.009067,0.005172
119,home_top_scorer_6,-0.008936,0.002867
134,home_top_assistor_6,-0.008541,0.003011
